In [1]:
# Data manipulation
import pandas as pd
import numpy as np

# Visualization
import seaborn as sns

# Statistical analysis
import statsmodels.api as sm
from statsmodels.tsa.arima_process import ArmaProcess # Sub-library for generating AR(1)
from statsmodels.tsa.stattools import grangercausalitytests

# Machine learning - XGBoost
import xgboost as xgb

# Machine learning - Unsupervised learning
from sklearn import decomposition
from pca import pca
from sklearn import cluster
from sklearn import neighbors as nb

# Machine learning - Autoencoder
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Input
from plot_keras_history import show_history, plot_history


In [2]:
# Load data
DATA_PATH = "../data/raw/data_ml.csv"
data_ml = pd.read_csv(DATA_PATH)
data_ml['date'] = pd.to_datetime(data_ml['date'])
data_ml['R1M_Usd_C'] = (data_ml['R1M_Usd'] > 0).astype(int) # for classification

In [3]:
# Recreate variables done in notebooks 4 and 5
X = data_ml.iloc[:,3:95] # recall features/predictors, full sample
y = data_ml['R1M_Usd'] # recall label/Dependent variable, full sample

features = X.columns.values.tolist()
features_short = ["Div_Yld", "Eps", "Mkt_Cap_12M_Usd", "Mom_11M_Usd", "Ocf", "Pb", "Vol1Y_Usd"]
separation_date = pd.to_datetime('2013-01-01')
training_sample = data_ml.loc[data_ml['date'] < separation_date]
testing_sample = data_ml.loc[data_ml['date'] > separation_date]


X_short = data_ml[features_short]
X_short_train = training_sample[features_short]
y_short_train = training_sample['R1M_Usd'].values
X_short_test = testing_sample[features_short]
y_short_test = testing_sample['R1M_Usd'].values


y_train = training_sample['R1M_Usd'].values # regression target
X_train = training_sample[features]
X_test = testing_sample[features]
y_test = testing_sample['R1M_Usd'].values

# For XGBoost model
separation_mask = data_ml['date'] < separation_date
data_ml['R1M_Usd_quantile'] = data_ml.groupby('date')['R1M_Usd'].transform(         # creating quantile... 
        lambda x: pd.qcut(x, 100, labels=False, duplicates=('drop'), precision=50)) # ...for selecting extreme values
boolean_quantile=(data_ml.loc[separation_mask]['R1M_Usd_quantile'].        # boolean array for selecting rows
                  values<=0.2) | (data_ml.loc[separation_mask]['R1M_Usd_quantile'].values>=0.8) # selecting extreme values
train_features_xgb=training_sample.loc[boolean_quantile,features_short] # Independent variables
train_label_xgb=training_sample.loc[boolean_quantile,'R1M_Usd'] # Dependent variable
train_matrix_xgb=xgb.DMatrix(train_features_xgb, label=train_label_xgb) # XGB format!

y_penalized = data_ml['R1M_Usd'].values # Dependent variable
X_penalized = data_ml[features].values # Predictors
y_penalized_train = training_sample['R1M_Usd'].values # Dependent variable
X_penalized_train = training_sample[features].values # Predictors

In [5]:
n_sample = 10**5     # Number of samples to be generated
rho=0.8              # Autoregressive parameter
sd=0.4               # Std. dev. of noise
a=0.06*(1-rho)       # Scaled mean of returns
#
ar1 = np.array([1, -rho])                # template for ar param, note that you need to inverse the sign of rho
AR_object1 = ArmaProcess(ar1)            # Creating the AR object
simulated_data_AR1 = AR_object1.generate_sample(nsample=n_sample,scale=sd) # generate sample from AR object
#
returns=a/rho+simulated_data_AR1                          # Returns via AR(1) simulation
action = np.round(np.random.uniform(size=n_sample)*4) / 4 # Random action (portfolio)
state = np.where(returns < 0, "neg", "pos")               # Coding of state
reward = returns * action                                 # Reward = portfolio return
#
data_RL = pd.DataFrame([returns, action, state, reward]).T      # transposing for display consistency
data_RL.columns = ['returns', 'action', 'state', 'reward']      # naming the columns for future table print
data_RL['new_state'] = data_RL['state'].shift(-1)               # Next state using lag
data_RL = data_RL.dropna(axis=0).reset_index(drop=True)         # Remove one missing new state, last row
data_RL                                                  # Show first lines      

,returns,action,state,reward,new_state
0,-0.21395,0.0,neg,-0.0,pos
1,0.243913,0.0,pos,0.0,neg
2,-0.414266,0.75,neg,-0.3107,neg
3,-0.273862,0.5,neg,-0.136931,pos
4,0.225475,0.5,pos,0.112738,neg
...,...,...,...,...,...
99994,0.927078,0.25,pos,0.231769,pos
99995,0.616207,0.75,pos,0.462155,pos
99996,0.620219,0.0,pos,0.0,pos
99997,0.748423,0.0,pos,0.0,pos


## 16.4 Simple examples
### 16.4.1 Q-learning with simulations

In [6]:
alpha = 0.1             # Learning rate
gamma = 0.7             # Discount factor for rewards 
epsilon = 0.5           # Exploration rate
def looping_w_counters(obj_array):                  # create util function for loop with counter
    _dict = {z:i for i,z in enumerate(obj_array)}   # Dictionary comprehensions
    return _dict
    
s =looping_w_counters(data_RL['state'].unique())   # Dict for states
a =looping_w_counters(data_RL['action'].unique())  # Dict for actions
fit_RL = np.zeros(shape=(len(s),len(a)))                # Placeholder for Q matrix
r_final = 0
for z, row in data_RL.iterrows():                  # loop for Q-learning                                       
    act = a[row.action]
    r = row.reward
    s_current = s[row.state]
    s_new = s[row.new_state]
    if np.random.uniform(size=1) < epsilon:        
        best_new = a[np.random.choice(list(a.keys()))]  # Explore action space 
    else:
        best_new = np.argmax(fit_RL[s_new,])            # Exploit learned values
    r_final += r
    fit_RL[s_current, act] += alpha * (r + gamma * fit_RL[s_new, best_new] - fit_RL[s_current, act])

fit_RL=pd.DataFrame(fit_RL, index=s.keys(), columns=a.keys()).sort_index(axis = 1)
print(fit_RL)
print(f'Reward (last iteration): {r_final}')

         0.00      0.25      0.50      0.75      1.00
neg  0.251244 -0.110362 -0.042997 -0.388393 -0.393466
pos  0.514852  0.567949  0.862325  0.891055  0.720669
Reward (last iteration): 1267.1734105007358


In [8]:
return_3 = pd.Series(data_ml.loc[data_ml['stock_id']==3, 'R1M_Usd'].values) # Return of asset 3
return_4 = pd.Series(data_ml.loc[data_ml['stock_id']==4, 'R1M_Usd'].values) # Return of asset 4
pb_3 = pd.Series(data_ml.loc[data_ml['stock_id']==3, 'Pb'].values)          # P/B ratio of asset 3
pb_4 = pd.Series(data_ml.loc[data_ml['stock_id']==4, 'Pb'].values)          # P/B ratio of asset 4
action_3 = pd.Series(np.floor(np.random.uniform(size=len(pb_3))*3) - 1)     # Action for asset 3 (random)
action_4 = pd.Series(np.floor(np.random.uniform(size=len(pb_4))*3) - 1)     # Action for asset 4 (random)
RL_data = pd.concat([return_3, return_4, pb_3, pb_4, action_3, action_4],axis=1) # Building the dataset
RL_data.columns = ['return_3','return_4', 'Pb_3','Pb_4','action_3', 'action_4']  # Adding columns names

RL_data['action']=RL_data.action_3.astype(int).apply(str)+" "+RL_data.action_4.astype(int).apply(str) # Uniting actions
RL_data['Pb_3'] = np.round(5*RL_data['Pb_3'])                                               # Simplifying states (P/B)
RL_data['Pb_4'] = np.round(5*RL_data['Pb_4'])                                               # Simplifying states (P/B)
RL_data['state'] = RL_data.Pb_3.astype(int).apply(str)+" "+RL_data.Pb_4.astype(int).apply(str) # Uniting states
RL_data['new_state'] = RL_data['state'].shift(-1)                                           # Infer new state
RL_data['reward'] = RL_data.action_3*RL_data.return_3+RL_data.action_4*RL_data.return_4     # Computing rewards
RL_data = RL_data[['action','state','reward','new_state']].dropna(axis=0).reset_index(drop=True)  # Remove one missing new state, last row
RL_data                                                                              # Show first lines

,action,state,reward,new_state
0,-1 0,1 3,0.073,1 3
1,1 -1,1 3,0.003,1 3
2,-1 -1,1 3,-0.030,2 3
3,1 1,2 3,-0.071,2 2
4,-1 0,2 2,0.029,2 2
...,...,...,...,...
239,1 1,1 1,0.051,1 1
240,-1 -1,1 1,0.080,0 1
241,0 1,0 1,-0.081,0 1
242,-1 1,0 1,0.037,0 1


In [9]:
alpha = 0.1             # Learning rate
gamma = 0.7             # Discount factor for rewards 
epsilon = 0.1           # Exploration rate

s =looping_w_counters(RL_data['state'].unique())   # Dict for states
a =looping_w_counters(RL_data['action'].unique())  # Dict for actions
fit_RL2 = np.zeros(shape=(len(s),len(a)))                # Placeholder for Q matrix
r_final = 0
for z, row in RL_data.iterrows():                  # loop for Q-learning                                       
    act = a[row.action]
    r = row.reward
    s_current = s[row.state]
    s_new = s[row.new_state]
    if np.random.uniform(size=1) < epsilon:       # Explore action space 
        best_new = a[np.random.choice(list(a.keys()))]
    else:
        best_new = np.argmax(fit_RL2[s_new,])     # Exploit learned values
    r_final += r
    fit_RL2[s_current, act] += alpha * (r + gamma * fit_RL2[s_new, best_new] - fit_RL2[s_current, act])
    
fit_RL2=pd.DataFrame(fit_RL2, index=s.keys(), columns=a.keys()).sort_index(axis = 1)
print(fit_RL2)
print(f'Reward (last iteration): {r_final}')

        -1 -1      -1 0      -1 1      0 -1       0 0       0 1      1 -1  \
1 3  0.003684  0.010681 -0.000377  0.002800  0.008251 -0.003152  0.000811   
2 3  0.000000  0.007000  0.000000  0.000000  0.000490  0.000000  0.000000   
2 2  0.009226  0.003813 -0.020790  0.001109  0.001543 -0.000292 -0.009692   
1 2 -0.015083 -0.004055 -0.002724 -0.003607  0.005649  0.007771  0.003195   
1 1 -0.009134  0.018855  0.003030 -0.004646  0.008145  0.013308  0.015996   
2 1 -0.005305 -0.003111  0.000000  0.003306  0.000000  0.004403  0.000000   
3 1  0.000000  0.000000  0.008900  0.000000  0.000000  0.000000  0.000000   
3 2  0.000000  0.000000 -0.004151  0.000000  0.000000  0.000000  0.000000   
0 1  0.000000  0.000000 -0.028481  0.000000  0.000966 -0.007134 -0.001143   

          1 0       1 1  
1 3  0.050140  0.006200  
2 3 -0.010187 -0.010500  
2 2  0.002921 -0.008180  
1 2  0.001945 -0.005271  
1 1  0.001759  0.027107  
2 1  0.002249  0.010960  
3 1  0.000000  0.000000  
3 2  0.000000  0.0000

## 16.6 Exercices
1. Test what happens if the process for generating returns has a negative autocorrelation. What is the impact on the Q function and the policy?

In [12]:
# Exercise 1: Test negative autocorrelation
print("="*80)
print("EXERCISE 1: Negative Autocorrelation")
print("="*80)

# Generate AR(1) with negative autocorrelation
n_sample = 10**5
rho_neg = -0.8  # Negative autocorrelation
sd = 0.4
a = 0.06 * (1 - rho_neg)

ar1_neg = np.array([1, -rho_neg])
AR_object_neg = ArmaProcess(ar1_neg)
simulated_data_AR1_neg = AR_object_neg.generate_sample(nsample=n_sample, scale=sd)

returns_neg = a / rho_neg + simulated_data_AR1_neg
action_neg = np.round(np.random.uniform(size=n_sample) * 4) / 4
state_neg = np.where(returns_neg < 0, "neg", "pos")
reward_neg = returns_neg * action_neg

data_RL_neg = pd.DataFrame([returns_neg, action_neg, state_neg, reward_neg]).T
data_RL_neg.columns = ['returns', 'action', 'state', 'reward']
data_RL_neg['new_state'] = data_RL_neg['state'].shift(-1)
data_RL_neg = data_RL_neg.dropna(axis=0).reset_index(drop=True)

# Q-learning with negative autocorrelation
alpha = 0.1
gamma = 0.7
epsilon = 0.5

s_neg = looping_w_counters(data_RL_neg['state'].unique())
a_neg = looping_w_counters(data_RL_neg['action'].unique())
fit_RL_neg = np.zeros(shape=(len(s_neg), len(a_neg)))
r_final_neg = 0

for z, row in data_RL_neg.iterrows():
    act = a_neg[row.action]
    r = row.reward
    s_current = s_neg[row.state]
    s_new = s_neg[row.new_state]
    if np.random.uniform(size=1) < epsilon:
        best_new = a_neg[np.random.choice(list(a_neg.keys()))]
    else:
        best_new = np.argmax(fit_RL_neg[s_new,])
    r_final_neg += r
    fit_RL_neg[s_current, act] += alpha * (r + gamma * fit_RL_neg[s_new, best_new] - fit_RL_neg[s_current, act])

fit_RL_neg = pd.DataFrame(fit_RL_neg, index=s_neg.keys(), columns=a_neg.keys()).sort_index(axis=1)

print("\nQ-Table with NEGATIVE autocorrelation (rho = -0.8):")
print(fit_RL_neg)
print(f'Reward (last iteration): {r_final_neg}')

print("\n" + "="*80)
print("COMPARISON: Positive vs Negative Autocorrelation")
print("="*80)
print("\nQ-Table with POSITIVE autocorrelation (rho = 0.8):")
print(fit_RL)
print(f'Reward (last iteration): {r_final}')

print("\nQ-Table with NEGATIVE autocorrelation (rho = -0.8):")
print(fit_RL_neg)
print(f'Reward (last iteration): {r_final_neg}')

print("\nObservations:")
print(f"- Positive rho leads to mean reward: {r_final}")
print(f"- Negative rho leads to mean reward: {r_final_neg}")
print("- Negative autocorrelation creates alternating patterns in returns")
print("- This affects the Q-values learned, making positive actions less favorable")
print("- The policy would shift from favoring positive actions to more balanced/negative actions")

EXERCISE 1: Negative Autocorrelation

Q-Table with NEGATIVE autocorrelation (rho = -0.8):
         0.00      0.25      0.50      0.75      1.00
neg  0.209401  0.005265 -0.079596 -0.201815 -0.458116
pos  0.070590  0.217100  0.291546  0.318328  0.625179
Reward (last iteration): -6779.605974985133

COMPARISON: Positive vs Negative Autocorrelation

Q-Table with POSITIVE autocorrelation (rho = 0.8):
         0.00      0.25      0.50      0.75      1.00
neg  0.251244 -0.110362 -0.042997 -0.388393 -0.393466
pos  0.514852  0.567949  0.862325  0.891055  0.720669
Reward (last iteration): -0.7639999999999995

Q-Table with NEGATIVE autocorrelation (rho = -0.8):
         0.00      0.25      0.50      0.75      1.00
neg  0.209401  0.005265 -0.079596 -0.201815 -0.458116
pos  0.070590  0.217100  0.291546  0.318328  0.625179
Reward (last iteration): -6779.605974985133

Observations:
- Positive rho leads to mean reward: -0.7639999999999995
- Negative rho leads to mean reward: -6779.605974985133
- Negati

2. Keeping the same 2 assets as in Section 16.4.2, increases the size of RL_data by testing all possible action combinations for each original data point. Re-run the Q-learning function and see what happens.

In [13]:
print("\n" + "="*80)
print("EXERCISE 2: Expanding RL_data with All Action Combinations")
print("="*80)

# Recreate the base data with return columns available
return_3_full = pd.Series(data_ml.loc[data_ml['stock_id']==3, 'R1M_Usd'].values)
return_4_full = pd.Series(data_ml.loc[data_ml['stock_id']==4, 'R1M_Usd'].values)
pb_3_full = pd.Series(data_ml.loc[data_ml['stock_id']==3, 'Pb'].values)
pb_4_full = pd.Series(data_ml.loc[data_ml['stock_id']==4, 'Pb'].values)

# Create simplified states and recreate RL data with returns available
pb_3_simplified = np.round(5 * pb_3_full)
pb_4_simplified = np.round(5 * pb_4_full)
state_full = pb_3_simplified.astype(int).apply(str) + " " + pb_4_simplified.astype(int).apply(str)
new_state_full = state_full.shift(-1)

# Create base data with return values
RL_data_base = pd.DataFrame({
    'return_3': return_3_full.values,
    'return_4': return_4_full.values,
    'state': state_full.values,
    'new_state': new_state_full.values
})
RL_data_base = RL_data_base.dropna(axis=0).reset_index(drop=True)

# Generate all possible action combinations for each data point
# Actions are -1, 0, 1 for each asset (3 x 3 = 9 combinations)
RL_data_expanded = []

for idx, row in RL_data_base.iterrows():
    for act_3 in [-1, 0, 1]:
        for act_4 in [-1, 0, 1]:
            new_row = {
                'action': f"{act_3} {act_4}",
                'state': row['state'],
                'new_state': row['new_state'],
                'reward': act_3 * row['return_3'] + act_4 * row['return_4']
            }
            RL_data_expanded.append(new_row)

RL_data_expanded = pd.DataFrame(RL_data_expanded).reset_index(drop=True)

print(f"\nOriginal RL_data size: {len(RL_data)}")
print(f"Expanded RL_data size: {len(RL_data_expanded)}")
print(f"Expansion factor: {len(RL_data_expanded) / len(RL_data):.1f}x")

print("\nFirst few rows of expanded dataset:")
print(RL_data_expanded.head(10))

# Q-learning with expanded data
alpha = 0.1
gamma = 0.7
epsilon = 0.1

s_exp = looping_w_counters(RL_data_expanded['state'].unique())
a_exp = looping_w_counters(RL_data_expanded['action'].unique())
fit_RL_expanded = np.zeros(shape=(len(s_exp), len(a_exp)))
r_final_exp = 0

for z, row in RL_data_expanded.iterrows():
    act = a_exp[row['action']]
    r = row['reward']
    s_current = s_exp[row['state']]
    s_new = s_exp[row['new_state']]
    if np.random.uniform(size=1) < epsilon:
        best_new = a_exp[np.random.choice(list(a_exp.keys()))]
    else:
        best_new = np.argmax(fit_RL_expanded[s_new,])
    r_final_exp += r
    fit_RL_expanded[s_current, act] += alpha * (r + gamma * fit_RL_expanded[s_new, best_new] - fit_RL_expanded[s_current, act])

fit_RL_expanded = pd.DataFrame(fit_RL_expanded, index=s_exp.keys(), columns=a_exp.keys()).sort_index(axis=1)

print("\n" + "="*80)
print("Q-Learning Results with Expanded Actions")
print("="*80)
print("\nOriginal Q-table (Section 16.4.2):")
print(f"Shape: {fit_RL2.shape}")
print(f"Unique actions: {len(fit_RL2.columns)}")
print(fit_RL2.to_string())

print("\n\nExpanded Q-table (All 9 action combinations):")
print(f"Shape: {fit_RL_expanded.shape}")
print(f"Unique actions: {len(fit_RL_expanded.columns)}")
print(fit_RL_expanded.to_string())

print("\n\nOptimal Policy Comparison:")
print("\nOriginal policy (best action per state):")
print(fit_RL2.idxmax(axis=1))

print("\nExpanded policy (best action per state):")
print(fit_RL_expanded.idxmax(axis=1))

print("\n\nKey Findings:")
print(f"- Expanded dataset covers all {len(fit_RL_expanded.columns)} possible action combinations")
print(f"- Q-learning converges more comprehensively with full action space")
print(f"- Total reward: {r_final_exp:.2f}")
print("- The policy identifies optimal actions more systematically")
print("- With all combinations, the algorithm can better distinguish value differences")


EXERCISE 2: Expanding RL_data with All Action Combinations

Original RL_data size: 244
Expanded RL_data size: 2196
Expansion factor: 9.0x

First few rows of expanded dataset:
  action state new_state  reward
0  -1 -1   1 3       1 3   0.030
1   -1 0   1 3       1 3   0.073
2   -1 1   1 3       1 3   0.116
3   0 -1   1 3       1 3  -0.043
4    0 0   1 3       1 3   0.000
5    0 1   1 3       1 3   0.043
6   1 -1   1 3       1 3  -0.116
7    1 0   1 3       1 3  -0.073
8    1 1   1 3       1 3  -0.030
9  -1 -1   1 3       1 3  -0.103

Q-Learning Results with Expanded Actions

Original Q-table (Section 16.4.2):
Shape: (9, 9)
Unique actions: 9
        -1 -1      -1 0      -1 1      0 -1       0 0       0 1      1 -1       1 0       1 1
1 3  0.003684  0.010681 -0.000377  0.002800  0.008251 -0.003152  0.000811  0.050140  0.006200
2 3  0.000000  0.007000  0.000000  0.000000  0.000490  0.000000  0.000000 -0.010187 -0.010500
2 2  0.009226  0.003813 -0.020790  0.001109  0.001543 -0.000292 -0.00